In [19]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from llama_index.core.tools import FunctionTool
from llama_index.core.tools import QueryEngineTool

from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    ReActAgent,
)

In [23]:
# -------------------------------------------------
# 1. Load documents
# -------------------------------------------------

documents = SimpleDirectoryReader(
    input_dir="./data"
).load_data()

print(f"Loaded {len(documents)} documents")
llm = Ollama(
    model="qwen3.5:9b",
    request_timeout=300.0,
    context_window=8192,
)

embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
)

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

chroma_collection = (
    chroma_client.get_or_create_collection(
        name="industrial_documents"
    )
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(
            chunk_size=512,
            chunk_overlap=50,
        ),
        embed_model,
    ],
    vector_store=vector_store,
)

nodes = pipeline.run(
    documents=documents,
    show_progress=True,
)

print(f"Created and stored {len(nodes)} nodes")

Loaded 1 documents


Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

2026-08-07 09:51:03,371 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-07 09:51:03,476 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Created and stored 11 nodes


In [24]:
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)

In [25]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=4,
    response_mode="compact",
)

In [26]:
response = query_engine.query(
    "Explain . What is weight."
)

print(response)


2026-08-07 09:51:49,716 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-07 09:53:30,352 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Weight is defined as a form of gravitational force acting upon an object's mass. It can be calculated by multiplying the mass ($m$) of the object by gravitational acceleration ($g$), expressed by the formula $W = mg$. Unlike mass, which utilizes kilograms in SI calculations, weight must be distinguished from it and typically expresses force measured in newtons. Practical instruments like load cells respond to this specific downward force rather than just a numeric value representing mass (for example, a 25 kg calibration mass exerts approximately 245 N of downward force). Additionally, density concepts differentiate between mass per unit volume and weight per unit volume, where the latter relies on multiplying the former by gravity.


In [4]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step, Event
 

class MyWorkflow(Workflow):
    @step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        # do something here
        return StopEvent(result="Hello, world!")


w = MyWorkflow(timeout=10, verbose=False)
result = await w.run()

In [6]:
from llama_index.core.workflow import Event

class ProcessingEvent(Event):
    intermediate_result: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> ProcessingEvent:
        # Process initial data
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()
result


'Finished processing: Step 1 complete'

In [11]:
from llama_index.core.workflow import Event
import random
from llama_index.utils.workflow import draw_all_possible_flows


class ProcessingEvent(Event):
    intermediate_result: str


class LoopEvent(Event):
    loop_output: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return ProcessingEvent(intermediate_result="First step complete.")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(verbose=False)
result = await w.run()
result

Good thing happened


'Finished processing: First step complete.'

In [12]:
draw_all_possible_flows(w, "flow.html")


flow.html


In [16]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
llm = Ollama(
    model="qwen3.5:9b",
    request_timeout=300.0,
    context_window=8192,
)

embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
)
# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

 
# we can pass functions directly without FunctionTool -- the fn/docstring are parsed for the name/description
multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],
    llm=llm,
)

addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],
    llm=llm,
)

# Create the workflow
workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
)

# Run the system
response = await workflow.run(user_msg="Can you add 5 and 3?")

In [18]:
print(response)

Yes, 5 plus 3 equals 8.


In [37]:
async def hello():
    return "Hellofdasfd"

In [38]:
x = await hello()
print(x)

Hellofdasfd
